# monsoonplus — training notebook

Trains **MonsoonPlusNet**, a trimodal gated-fusion graph neural network that forecasts
Chennai road speeds at t+15 / t+30 / t+60 minutes.

**Run this top to bottom.** Runtime → Run all. Nothing needs editing, no API key is
required, and it works on the free CPU tier as well as the free GPU tier.

| Tier | Expected total time |
| --- | --- |
| Free GPU (T4) | ~8–12 minutes |
| Free CPU | ~25–35 minutes |

**To pick a GPU:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.
You do not have to; the notebook detects what it has and adjusts.

At the end you download four JSON files and drop them into `frontend/public/data/`.
That is the only handoff between this notebook and the website.

## 1. Get the code

Clone your fork, or use the local copy if you uploaded the folder to Colab instead.
Edit `REPO_URL` to point at your own GitHub repository.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/monsoonplus.git"  # <-- change this
REPO_DIR = "/content/monsoonplus"

import os, sys, subprocess, pathlib

if not os.path.exists(REPO_DIR):
    # If you uploaded the folder manually instead of cloning, just make sure it sits
    # at /content/monsoonplus and this cell will skip the clone.
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

if not os.path.exists(REPO_DIR):
    raise SystemExit(
        "Could not find the project.\n"
        "Either set REPO_URL to your repository, or upload the monsoonplus folder\n"
        "to /content/monsoonplus using the file browser on the left."
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("working directory:", os.getcwd())
print(sorted(os.listdir()))

## 2. Install dependencies

Colab already ships PyTorch, so only PyTorch Geometric and h5py usually need installing.
`torch_geometric` is pure Python for the layers we use (GCNConv, GATConv) — no
`torch-scatter` compile step, which is what used to make this slow and fragile.

In [ ]:
%pip install -q torch_geometric h5py

import torch
print("torch       :", torch.__version__)
print("CUDA avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("GPU         : none — running on CPU, which is supported but slower")

import torch_geometric
print("torch_geom  :", torch_geometric.__version__)

## 3. Check the road network

`config/segments.json` is the single source of truth for the 10 road segments and 8
junctions. The model, the backend, the collector and the website all read this one
file. If this cell prints `config valid`, everything downstream agrees about the map.

In [ ]:
!python -m config.segments

## 4. Download METR-LA (optional but recommended)

METR-LA is 207 loop detectors on Los Angeles freeways at 5-minute resolution — the
standard benchmark for traffic forecasting. We use it for **phase-1 pretraining**:
the traffic encoder learns general temporal dynamics from a large real dataset before
ever seeing Chennai. That is the transfer-learning part of the novelty claim.

If the download fails (mirrors move), the next cells fall back to a clearly-labelled
synthetic stand-in and everything still runs. Nothing breaks.

In [ ]:
from ml.data.metr_la import download_metr_la, load_metr_la, to_traffic_features

download_metr_la()
data = load_metr_la()

print()
print("label      :", data.label, "(is_real =", data.is_real, ")")
print("source     :", data.source)
print("shape      :", data.speeds.shape, "(timesteps, sensors)")
print("observed   : %.1f%% of cells" % (100 * (data.speeds > 0).mean()))

features = to_traffic_features(data.speeds)
print("features   :", features.shape, "(speed, free-flow proxy, confidence)")

### Why `h5py` and not `pandas.read_hdf`

`metr-la.h5` is an old pandas *fixed-format* block whose axis labels are byte strings.
`pandas.read_hdf` needs the `pytables` package (often unavailable) and, once installed,
raises a `TypeError` decoding those labels on pandas 3.x. We read the HDF5 datasets
directly with `h5py` and decode the labels ourselves, so neither problem can happen.

## 5. Inspect the Chennai window

There is no public dataset of Chennai road speeds during monsoon, and a student cannot
collect months of it before a deadline. So we generate one from an explicit physical
process and **label it synthetic everywhere it appears**.

The design point to notice: NDWI tracks *accumulated* wetness with a long decay, and
the flooding term depends on accumulation rather than on instantaneous rainfall. That
is what gives the satellite modality information the weather history genuinely lacks —
without it, "trimodal" would be decoration.

In [ ]:
!python -m ml.data.chennai_synthetic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ml.data.chennai_synthetic import generate_chennai_window

window = generate_chennai_window(days=45)
hours = 24 * 6  # six days
steps = hours * 12
t = np.arange(steps) * 5 / 60.0

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, window.rain[:steps].mean(axis=1), color="#1F5C8C")
axes[0].set_ylabel("rain mm/h"); axes[0].set_title("Synthetic Chennai window (first 6 days)")
axes[1].plot(t, window.wetness[:steps].mean(axis=1), color="#6B3FA0")
axes[1].set_ylabel("wetness 0-1")
axes[2].plot(t, window.speeds[:steps].mean(axis=1), color="#1F5C47")
axes[2].set_ylabel("speed km/h"); axes[2].set_xlabel("hours")
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print("Notice the lag: wetness keeps rising after the rain peaks, and drains slowly.")
print("That lag is exactly what the satellite branch contributes.")

## 6. Check the graph

Two road segments are connected when they share a junction. Edge weights come from a
thresholded Gaussian kernel over the distance between segment midpoints.

**Why this matters:** `GCNConv` adds self-loops with weight 1.0. An earlier version
weighted edges by `1/travel_time`, giving values around 0.003 — after normalisation,
a node's own features carried over 99% of the message and the "GNN" was really a
per-node MLP. The Gaussian kernel puts neighbour weights in a 0.1–0.7 band instead.
`assert_graph_contributes()` is the regression guard.

In [ ]:
!python -m ml.data.graph

## 7. Train

Four phases:

| Phase | Data | What it does |
| --- | --- | --- |
| 1 | real METR-LA | pretrain the traffic branch (weather + satellite masked **absent**, not zero-filled) |
| 2 | synthetic Chennai | transfer; all three modalities now live |
| 3 | synthetic Chennai | cost-sensitive fine-tune — rain ×3, heavy rain ×8 |
| 4 | real collected logs | skipped unless `logs/` has data |

Every phase keeps the checkpoint with the lowest **rain-weighted** validation MAE, not
the lowest plain validation MAE. About 60% of the window is clear weather, so plain
validation MAE would pick whichever epoch is best at dry days — the opposite of the point.

Swap the graph layer with `--graph-layer gat`. That is the whole change.

In [ ]:
!python -m ml.training.train --graph-layer gcn

## 8. Evaluate honestly

This runs the model and **all three baselines** (persistence, historical average, an
LSTM with no graph) through the same code path on the same test split.

The rule the project holds itself to: an aggregate MAE is never reported without the
rain-vs-clear split next to it. An overall "X% better" number can be driven entirely by
clear weather while the rain error gets worse, and `honesty_notes` in the output is
generated from the numbers themselves to say so when that happens.

In [ ]:
!python -m ml.training.evaluate

## 9. Ablations

Two questions, and both answers get reported whichever way they come out:

1. **GCN vs GAT** — does the graph layer choice matter on a 10-node graph?
2. **Modality drop** — is this genuinely trimodal, or a traffic model with decoration?
   Traffic-only vs traffic+weather vs all three, using the availability mask (not
   zero-filling, which would still feed the encoder a constant signal).

This cell takes a few minutes. Skip it if you are short on time — nothing downstream
depends on it.

In [ ]:
!python -m ml.training.ablations --epochs 8 --days 30

## 10. Does the gate actually do what we claim?

The central claim is: *in heavy rain the model leans more on weather and satellite, and
less on traffic history.* The gate is readable from outside, so this is checkable rather
than rhetorical. If the numbers do not support the claim, they are printed anyway.

In [ ]:
import json, pathlib

report_path = pathlib.Path("ml/reports/eval_synthetic_chennai.json")
report = json.loads(report_path.read_text())

gate = report["gate_by_regime"]
print(f"{'regime':<12}{'traffic':>9}{'weather':>9}{'satellite':>11}")
for regime, weights in gate.items():
    print(f"{regime:<12}{weights['traffic']:>8.1f}%{weights['weather']:>8.1f}%{weights['satellite']:>10.1f}%")

claim = report["gate_shift_claim"]
print()
print("claim supported:", claim["supported"])
print(claim["reason"])

print("\nHonest reading of the error numbers:")
for note in report["honesty_notes"]:
    print(" -", note)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes = ["clear", "rain", "heavy_rain"]
labels = ["Clear", "Rain", "Heavy rain"]
traffic = [gate[r]["traffic"] for r in regimes]
weather = [gate[r]["weather"] for r in regimes]
satellite = [gate[r]["satellite"] for r in regimes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(3)
axes[0].bar(x, traffic, label="Traffic", color="#1F5C47")
axes[0].bar(x, weather, bottom=traffic, label="Weather", color="#1F5C8C")
axes[0].bar(x, satellite, bottom=np.add(traffic, weather), label="Satellite", color="#6B3FA0")
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("gate weight %"); axes[0].set_title("What the model leans on")
axes[0].legend(fontsize=8)

model_mae = [report["model"]["mae_by_label"].get(r) for r in regimes]
persistence = next(b for b in report["baselines"] if b["name"] == "persistence")
base_mae = [persistence["mae_by_label"].get(r) for r in regimes]
w = 0.38
axes[1].bar(x - w/2, model_mae, w, label="monsoonplus", color="#1F5C47")
axes[1].bar(x + w/2, base_mae, w, label="persistence", color="#B87D04")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("MAE km/h"); axes[1].set_title("Error by weather regime")
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(alpha=0.25, axis="y")
plt.tight_layout(); plt.show()

## 11. Export the files the website reads

This is the handoff. The deployed site is static: it reads these JSON files and needs
no backend at all.

In [ ]:
!python -m ml.export.export_predictions

In [ ]:
import json, pathlib

out = pathlib.Path("frontend/public/data")
for path in sorted(out.glob("*.json")):
    payload = json.loads(path.read_text())  # proves it parses
    print(f"{path.name:<42}{path.stat().st_size/1024:>8.1f} KB   keys: {len(payload)}")

# Hard check: no NaN survived into any file. NaN is not valid JSON and would make
# JSON.parse throw in the browser.
for path in out.glob("*.json"):
    text = path.read_text()
    assert "NaN" not in text and "Infinity" not in text, f"{path.name} contains NaN/Infinity"
print("\nAll files parse and contain no NaN.")

## 12. Download

Run this, then copy the four files into `frontend/public/data/` in your local
repository, **overwriting** what is there. Commit them — they are the deployed site's
entire data layer.

The trained checkpoint is also downloaded. It is gitignored by default (it is a binary
and the website does not need it), but keep it if you want to run the local backend or
the collector without retraining.

In [ ]:
import shutil, pathlib

try:
    from google.colab import files  # type: ignore
    in_colab = True
except ImportError:
    in_colab = False

wanted = [
    "frontend/public/data/predictions_synthetic_chennai.json",
    "frontend/public/data/predictions_metr_la.json",
    "frontend/public/data/predictions_live_template.json",
    "frontend/public/data/model_card.json",
]

# A single zip is far less fiddly than four separate download prompts.
bundle = pathlib.Path("/content/monsoonplus_data")
bundle.mkdir(exist_ok=True)
for item in wanted:
    shutil.copy(item, bundle / pathlib.Path(item).name)

checkpoint = pathlib.Path("ml/checkpoints/monsoonplus_final.pt")
if checkpoint.exists():
    shutil.copy(checkpoint, bundle / checkpoint.name)
for report in pathlib.Path("ml/reports").glob("*.json"):
    shutil.copy(report, bundle / report.name)

archive = shutil.make_archive("/content/monsoonplus_outputs", "zip", bundle)
print("bundled:", archive)
for item in sorted(bundle.iterdir()):
    print(f"  {item.name:<42}{item.stat().st_size/1024:>8.1f} KB")

if in_colab:
    files.download(archive)
else:
    print("\nNot running in Colab — the zip is at", archive)

## 13. What to do with the download

Unzip `monsoonplus_outputs.zip`, then:

1. Copy these four files into `frontend/public/data/`, overwriting the existing ones:
   - `predictions_synthetic_chennai.json`
   - `predictions_metr_la.json`
   - `predictions_live_template.json`
   - `model_card.json`
2. (Optional) copy `monsoonplus_final.pt` into `ml/checkpoints/` if you want the local
   backend or collector.
3. Run the site:
   ```bash
   cd frontend
   npm install
   npm run dev
   ```
4. Commit and push the four JSON files. Vercel redeploys automatically.

**Do not commit** `.env`, API keys, or `ml/data/raw/`. `.gitignore` already blocks them.

---

### Reminder on what these numbers are

The Chennai results come from a **synthetic** window generated by an explicit physical
process, not from measured Chennai traffic. The METR-LA results are from real data. The
satellite branch trains on synthetic NDWI unless you supply a real Earth Engine export
(README section 7). All of this is stated on the site itself, in Model Lab and in
Data & System — say it the same way in your viva and the work holds up.